# Speech Emotion Recognition (SER) High-Accuracy (~90%+) Production Pipeline
**Author:** Shahzaib Hussain | **Final Year Project (FYP)**  
**Datasets:** CREMA-D (7,442), RAVDESS (1,440), SAVEE (480), TESS (2,800) $\rightarrow$ **12,162 Voice Recordings (Full Uncapped Corpus + Acoustic Augmentation)**  
**Methodology:** 430-Dim Handcrafted Log-Mel/Cepstral/Spectral/Teager Extractor + GPU Wav2Vec2 SSL Embeddings + Corpus-Aware Normalization + Validation-Optimized Stacking & Soft-Voting Ensemble  
**Production Artifact:** Single unified production bundle (`final_ser_production_bundle.joblib`) with real-time inference (`RTF << 1.0`)

## 1. Environment Setup, Imports & Configuration Loader

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP, IMPORTS & CONFIGURATION LOADER
# ==============================================================================
import os
import re
import glob
import time
import json
import yaml
import random
import warnings
from typing import Dict, List, Tuple, Any, Optional

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.signal
import scipy.stats
import soundfile as sf
import joblib

import torch
import torchaudio
import torchaudio.transforms as AT
import torchaudio.functional as AF

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, log_loss
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb

def load_ser_config(config_path: str = "ser_config.yaml") -> Dict[str, Any]:
    candidates = [
        config_path,
        os.path.join(r"C:\Users\Shahzaib Hussain\Desktop\Shahzaib Hussain\FYP\Voice and Facial recognition", config_path)
    ]
    for c in candidates:
        if os.path.exists(c):
            with open(c, "r", encoding="utf-8") as f:
                return yaml.safe_load(f)
    raise FileNotFoundError(f"Configuration file not found in candidates: {candidates}")

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

CFG = load_ser_config("ser_config.yaml")
seed_everything(CFG["environment"]["seed"])

os.makedirs(CFG["environment"]["output_dir"], exist_ok=True)
os.makedirs(CFG["environment"]["checkpoint_dir"], exist_ok=True)

DEVICE = torch.device("cuda" if (CFG["environment"].get("device", "cuda") == "cuda" and torch.cuda.is_available()) else "cpu")

print(f"[INFO] Loaded configuration for: '{CFG['environment']['project_name']}'")
print(f"[INFO] Output Directory: {CFG['environment']['output_dir']}")
print(f"[INFO] Final Bundle Target: {CFG['outputs']['final_bundle_path']}")
print(f"[INFO] Hardware Acceleration: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


## 2. Multi-Corpus Dataset Indexing (CREMA-D, RAVDESS, SAVEE, TESS)

In [ ]:
# ==============================================================================
# 2. MULTI-CORPUS DATASET PARTITIONING (UNCAPPED & STRATIFIED DOMAIN-ALIGNED)
# Indexes CREMA-D (7,442), RAVDESS (1,440), SAVEE (480), and TESS (2,800)
# ==============================================================================
EMOTION_TO_IDX = {v: int(k) for k, v in CFG["taxonomy"]["class_names"].items()}
IDX_TO_EMOTION = {int(k): v for k, v in CFG["taxonomy"]["class_names"].items()}

CREMA_MAP = {"NEU": "neutral", "HAP": "happy", "SAD": "sad", "ANG": "angry", "FEA": "fear", "DIS": "disgust"}
RAV_MAP = {
    "01": "neutral", "02": "neutral", # Calm merged into Neutral
    "03": "happy", "04": "sad", "05": "angry",
    "06": "fear", "07": "disgust", "08": "surprise"
}
SAVEE_MAP = {
    "n": "neutral", "h": "happy", "sa": "sad", "a": "angry",
    "f": "fear", "d": "disgust", "su": "surprise"
}
TESS_MAP = {
    "neutral": "neutral", "happy": "happy", "sad": "sad",
    "angry": "angry", "fear": "fear", "disgust": "disgust", "ps": "surprise"
}

def index_all_speech_datasets(cfg: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    allowed_emotions = set(cfg["taxonomy"]["class_names"].values())
    seed = cfg["environment"]["seed"]
    records = []

    # 1. CREMA-D (Speaker-Independent)
    c_cfg = cfg["datasets"]["crema_d"]
    if c_cfg.get("enabled", True):
        for p in glob.glob(os.path.join(c_cfg["audio_dir"], "*.wav")):
            parts = os.path.basename(p).replace(".wav", "").split("_")
            if len(parts) >= 3 and parts[2] in CREMA_MAP:
                emo = CREMA_MAP[parts[2]]
                if emo in allowed_emotions:
                    spk = int(parts[0])
                    split = "train" if spk <= c_cfg["train_speakers_end"] else ("val" if spk <= c_cfg["val_speakers_end"] else "test")
                    records.append({"filepath": p, "dataset": "CREMA-D", "speaker_id": f"CREMA_{spk}", "emotion": emo, "split": split})

    # 2. RAVDESS (Speaker-Independent)
    r_cfg = cfg["datasets"]["ravdess"]
    if r_cfg.get("enabled", True):
        for p in glob.glob(os.path.join(r_cfg["audio_dir"], "**", "*.wav"), recursive=True):
            parts = os.path.basename(p).replace(".wav", "").split("-")
            if len(parts) >= 7 and parts[2] in RAV_MAP:
                emo = RAV_MAP[parts[2]]
                if emo in allowed_emotions:
                    actor_id = int(parts[6])
                    split = "train" if actor_id <= r_cfg["train_actors_end"] else ("val" if actor_id <= r_cfg["val_actors_end"] else "test")
                    records.append({"filepath": p, "dataset": "RAVDESS", "speaker_id": f"RAV_{actor_id:02d}", "emotion": emo, "split": split})

    # 3. SAVEE
    s_cfg = cfg["datasets"]["savee"]
    if s_cfg.get("enabled", True):
        savee_holdout = []
        for p in glob.glob(os.path.join(s_cfg["audio_dir"], "**", "*.wav"), recursive=True):
            m = re.match(r"([A-Za-z]+)_([a-zA-Z]+)(\d+)", os.path.basename(p).replace(".wav", ""))
            if m:
                spk = m.group(1).upper()
                code = m.group(2).lower()
                emo = SAVEE_MAP.get(code)
                if emo in allowed_emotions:
                    if spk in s_cfg["train_speakers"]:
                        records.append({"filepath": p, "dataset": "SAVEE", "speaker_id": f"SAVEE_{spk}", "emotion": emo, "split": "train"})
                    else:
                        savee_holdout.append({"filepath": p, "dataset": "SAVEE", "speaker_id": f"SAVEE_{spk}", "emotion": emo, "split": "test"})
        if savee_holdout:
            s_df = pd.DataFrame(savee_holdout)
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
            for val_i, test_i in sss.split(s_df, s_df["emotion"]):
                s_df.iloc[val_i, s_df.columns.get_loc("split")] = "val"
                s_df.iloc[test_i, s_df.columns.get_loc("split")] = "test"
            records.extend(s_df.to_dict("records"))

    # 4. TESS (Stratified across both speakers YAF & OAF to prevent single-speaker age collapse)
    t_cfg = cfg["datasets"]["tess"]
    if t_cfg.get("enabled", True):
        tess_records = []
        for p in glob.glob(os.path.join(t_cfg["audio_dir"], "**", "*.wav"), recursive=True):
            parts = os.path.basename(p).replace(".wav", "").split("_")
            if len(parts) >= 3:
                spk = "OAF" if parts[0].upper().startswith("OA") else parts[0].upper()
                code = parts[2].lower()
                emo = TESS_MAP.get(code)
                if emo in allowed_emotions:
                    tess_records.append({"filepath": p, "dataset": "TESS", "speaker_id": f"TESS_{spk}", "emotion": emo, "split": "train"})
        if tess_records:
            t_df = pd.DataFrame(tess_records)
            strat_key = t_df["speaker_id"] + "_" + t_df["emotion"]
            val_test_ratio = t_cfg.get("val_ratio", 0.10) + t_cfg.get("test_ratio", 0.15)
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=val_test_ratio, random_state=seed)
            for tr_i, vt_i in sss1.split(t_df, strat_key):
                t_df.iloc[tr_i, t_df.columns.get_loc("split")] = "train"
                vt_sub = t_df.iloc[vt_i].copy()
                rel_test = t_cfg.get("test_ratio", 0.15) / max(val_test_ratio, 1e-6)
                sss2 = StratifiedShuffleSplit(n_splits=1, test_size=rel_test, random_state=seed)
                for v_i, te_i in sss2.split(vt_sub, vt_sub["speaker_id"] + "_" + vt_sub["emotion"]):
                    t_df.iloc[vt_i[v_i], t_df.columns.get_loc("split")] = "val"
                    t_df.iloc[vt_i[te_i], t_df.columns.get_loc("split")] = "test"
            records.extend(t_df.to_dict("records"))

    df = pd.DataFrame(records).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "val"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = index_all_speech_datasets(CFG)

print("="*75)
print("MULTI-CORPUS PARTITIONING SUMMARY (FULL UNCAPPED DATASET):")
print(f"  • Training   : {len(train_df):,} files | By Dataset: {train_df['dataset'].value_counts().to_dict()}")
print(f"  • Validation : {len(val_df):,} files | By Dataset: {val_df['dataset'].value_counts().to_dict()}")
print(f"  • Held-Out   : {len(test_df):,} files | By Dataset: {test_df['dataset'].value_counts().to_dict()}")
print(f"  • Total Audio Recordings Indexed: {len(train_df) + len(val_df) + len(test_df):,}")
print("="*75)


## 3. Audio Preprocessor with Pre-Emphasis, VAD & Acoustic Augmentation

In [ ]:
# ==============================================================================
# 3. AUDIO PREPROCESSOR WITH PRE-EMPHASIS, NORMALIZATION, VAD & AUGMENTATION
# ==============================================================================
class AudioPreprocessor:
    """
    Standardizes speech recordings across multi-corpus boundaries:
    - Monophonic conversion & 16 kHz resampling
    - Energy-based Voice Activity Detection (VAD) with soft margin
    - Pre-emphasis filtering (boosts high-frequency formants)
    - Waveform amplitude z-normalization
    - Optional acoustic data augmentation (AWGN + pitch/speed perturbation)
    """
    def __init__(self, cfg: Dict[str, Any]):
        self.cfg = cfg
        self.target_sr = cfg["audio"]["sample_rate"]
        self.min_duration = cfg["audio"]["min_duration_sec"]
        self.vad_enabled = cfg["audio"]["vad"]["enabled"]
        self.energy_thresh_ratio = cfg["audio"]["vad"]["energy_threshold_ratio"]
        self.normalize_waveform = cfg["audio"].get("normalize_waveform", True)
        self.pre_emphasis = cfg["audio"].get("pre_emphasis_coeff", 0.97)

    def process_file(self, audio_path: str) -> Tuple[Optional[np.ndarray], int, Dict[str, float]]:
        if not os.path.exists(audio_path):
            return None, self.target_sr, {}

        try:
            y, sr = sf.read(audio_path, dtype="float32")
        except Exception:
            return None, self.target_sr, {}

        if y.ndim > 1:
            y = np.mean(y, axis=1)

        if sr != self.target_sr:
            y_tensor = torch.from_numpy(y).unsqueeze(0)
            y = AF.resample(y_tensor, sr, self.target_sr).squeeze(0).numpy()
            sr = self.target_sr

        original_dur = len(y) / sr
        if original_dur < self.min_duration:
            return None, sr, {"original_duration": original_dur}

        # Energy-based Voice Activity Detection (VAD)
        if self.vad_enabled and len(y) > int(0.06 * sr):
            frame_len = int(self.cfg["audio"]["vad"]["frame_len_sec"] * sr)
            hop_len = int(self.cfg["audio"]["vad"]["hop_len_sec"] * sr)
            if len(y) >= frame_len:
                frames = np.lib.stride_tricks.sliding_window_view(y, frame_len)[::hop_len]
                energies = np.sum(frames ** 2, axis=-1)
                if len(energies) > 0 and np.max(energies) > 1e-6:
                    threshold = self.energy_thresh_ratio * np.max(energies)
                    active_frames = np.where(energies > threshold)[0]
                    if len(active_frames) > 0:
                        pad_frames = 2
                        start_f = max(0, active_frames[0] - pad_frames)
                        end_f = min(len(energies) - 1, active_frames[-1] + pad_frames)
                        start_sample = start_f * hop_len
                        end_sample = min(len(y), (end_f + 1) * hop_len + frame_len)
                        y_active = y[start_sample:end_sample]
                    else:
                        y_active = y
                else:
                    y_active = y
            else:
                y_active = y
        else:
            y_active = y

        # Standardize waveform amplitude
        if self.normalize_waveform and len(y_active) > 0:
            peak = np.max(np.abs(y_active))
            if peak > 1e-6:
                y_active = y_active / peak
            std_val = float(np.std(y_active))
            if std_val > 1e-6:
                y_active = (y_active - np.mean(y_active)) / std_val
        active_dur = len(y_active) / sr
        stats = {
            "original_duration": round(original_dur, 3),
            "active_duration": round(active_dur, 3),
            "speech_ratio": round(active_dur / max(original_dur, 1e-4), 3)
        }
        return y_active.astype(np.float32), sr, stats

    def augment_waveform(self, y: np.ndarray, seed_offset: int = 0) -> np.ndarray:
        """
        Applies mild speed/time-stretch perturbation + low-level Gaussian noise
        to strengthen classifier decision boundaries without distorting emotion.
        """
        rng = np.random.default_rng(self.cfg["environment"]["seed"] + seed_offset)
        rate = rng.uniform(0.94, 1.06)
        new_len = max(int(len(y) / rate), int(0.4 * self.target_sr))
        y_aug = scipy.signal.resample(y, new_len).astype(np.float32)
        noise_amp = rng.uniform(0.002, 0.012) * np.std(y_aug)
        y_aug = y_aug + rng.normal(0.0, noise_amp, size=y_aug.shape).astype(np.float32)
        std_val = float(np.std(y_aug))
        if std_val > 1e-6:
            y_aug = (y_aug - np.mean(y_aug)) / std_val
        return y_aug

preprocessor = AudioPreprocessor(CFG)
sample_test_path = train_df["filepath"].iloc[0]
sample_y, sample_sr, sample_dur_stats = preprocessor.process_file(sample_test_path)
print(f"[TEST] Preprocessed sample recording from '{train_df['dataset'].iloc[0]}':")
print(f"       File: {os.path.basename(sample_test_path)}")
print(f"       Active duration: {sample_dur_stats['active_duration']}s (original: {sample_dur_stats['original_duration']}s)")


## 4. High-Resolution Acoustic Feature Extractor + GPU Wav2Vec2 SSL Embedder

In [ ]:
# ==============================================================================
# 4. HIGH-RESOLUTION ACOUSTIC EXTRACTOR (430+ HANDCRAFTED + GPU WAV2VEC2 SSL)
# ==============================================================================
class ComprehensiveAcousticExtractor:
    """
    Extracts rich statistical functionals across:
    - Prosodic RMS Energy, Teager Energy Operator (TEO), Speaking Rate & Voiced Ratio
    - Pitch (F0) Contour Statistics (mean, std, min, max, range, q10, q90, slope, iqr)
    - Voice Quality (Harmonics-to-Noise Ratio, Local Jitter, Local Shimmer)
    - Spectral Moments (Centroid, Bandwidth, 85%/95% Rolloff, Flatness, Flux, 6-Band Contrast)
    - 40 MFCCs (mean, std, q10, q90, skew) + 40 Delta + 40 Delta-Delta
    - 64 Log-Decibel Mel-Filterbank Energies (mean, std, q90)
    """
    def __init__(self, cfg: Dict[str, Any]):
        self.cfg = cfg
        self.sr = cfg["audio"]["sample_rate"]
        self.pre_emph = cfg["audio"].get("pre_emphasis_coeff", 0.97)
        hop_len = int(cfg["features"]["prosody"]["hop_len_sec"] * self.sr)
        n_fft = cfg["features"]["spectral"]["n_fft"]
        self.n_mfcc = cfg["features"]["mfcc"].get("n_mfcc", 40)
        self.n_mels = cfg["features"].get("mel_spectrogram", {}).get("n_mels", 64)

        self.mfcc_transform = AT.MFCC(
            sample_rate=self.sr,
            n_mfcc=self.n_mfcc,
            melkwargs={
                "n_fft": n_fft,
                "hop_length": hop_len,
                "n_mels": cfg["features"]["mfcc"].get("n_mels", 80)
            }
        )
        self.mel_transform = AT.MelSpectrogram(
            sample_rate=self.sr,
            n_fft=n_fft,
            hop_length=hop_len,
            n_mels=self.n_mels
        )
        self.amp_to_db = AT.AmplitudeToDB(stype="power", top_db=80.0)

    def extract_all_handcrafted(self, y: np.ndarray, dur_stats: Optional[Dict[str, float]] = None) -> Dict[str, float]:
        sr = self.sr
        feats: Dict[str, float] = {}

        # Duration & Temporal Ratios
        feats["TEMP_active_dur"] = float(dur_stats.get("active_duration", len(y) / sr)) if dur_stats else float(len(y) / sr)
        feats["TEMP_speech_ratio"] = float(dur_stats.get("speech_ratio", 1.0)) if dur_stats else 1.0

        # Pre-emphasized signal for spectral/cepstral clarity
        y_emph = np.append(y[0], y[1:] - self.pre_emph * y[:-1]).astype(np.float32)

        # 1. Framing
        frame_len = int(self.cfg["features"]["prosody"]["frame_len_sec"] * sr)
        hop_len = int(self.cfg["features"]["prosody"]["hop_len_sec"] * sr)
        if len(y) < frame_len:
            y = np.pad(y, (0, frame_len - len(y)))
            y_emph = np.pad(y_emph, (0, frame_len - len(y_emph)))

        strided = np.lib.stride_tricks.sliding_window_view(y, frame_len)[::hop_len]
        strided_emph = np.lib.stride_tricks.sliding_window_view(y_emph, frame_len)[::hop_len]

        # 2. Prosodic RMS Energy & Teager Energy Operator (TEO)
        rms = np.sqrt(np.mean(strided ** 2, axis=-1) + 1e-9)
        q10, q25, q75, q90 = np.percentile(rms, [10, 25, 75, 90])
        feats["PROS_RMS_mean"] = float(np.mean(rms))
        feats["PROS_RMS_std"] = float(np.std(rms))
        feats["PROS_RMS_max"] = float(np.max(rms))
        feats["PROS_RMS_q10"] = float(q10)
        feats["PROS_RMS_q25"] = float(q25)
        feats["PROS_RMS_q75"] = float(q75)
        feats["PROS_RMS_q90"] = float(q90)
        feats["PROS_RMS_iqr"] = float(q75 - q25)
        feats["PROS_RMS_skew"] = float(scipy.stats.skew(rms)) if len(rms) > 2 else 0.0

        # Teager Energy Operator (captures non-linear vocal fold tension in anger/fear)
        if len(y) > 3:
            teo = y[1:-1] ** 2 - y[:-2] * y[2:]
            feats["PROS_TEO_mean"] = float(np.mean(teo))
            feats["PROS_TEO_std"] = float(np.std(teo))
            feats["PROS_TEO_max"] = float(np.max(teo))
            feats["PROS_TEO_q90"] = float(np.percentile(teo, 90))
        else:
            for k in ["PROS_TEO_mean", "PROS_TEO_std", "PROS_TEO_max", "PROS_TEO_q90"]:
                feats[k] = 0.0

        # 3. Zero Crossing Rate (ZCR)
        zcr = np.mean(np.abs(np.diff(np.sign(strided))), axis=-1) / 2.0
        feats["SPEC_ZCR_mean"] = float(np.mean(zcr))
        feats["SPEC_ZCR_std"] = float(np.std(zcr))
        feats["SPEC_ZCR_q90"] = float(np.percentile(zcr, 90))

        # 4. High-Resolution Spectral Moments, Flatness, Flux & 6-Subband Contrast
        n_fft = self.cfg["features"]["spectral"]["n_fft"]
        win = np.hanning(frame_len)
        fft_spec = np.abs(np.fft.rfft(strided_emph * win, n=n_fft, axis=-1)) + 1e-9
        freqs = np.fft.rfftfreq(n_fft, 1.0 / sr)
        spec_sum = np.sum(fft_spec, axis=-1, keepdims=True)

        centroid = np.sum(freqs * fft_spec, axis=-1, keepdims=True) / spec_sum
        bandwidth = np.sqrt(np.sum(((freqs - centroid) ** 2) * fft_spec, axis=-1) / spec_sum.squeeze(-1) + 1e-9)
        flatness = np.exp(np.mean(np.log(fft_spec), axis=-1)) / (np.mean(fft_spec, axis=-1) + 1e-9)

        norm_spec = fft_spec / spec_sum
        entropy = -np.sum(norm_spec * np.log2(norm_spec + 1e-12), axis=-1)

        if len(fft_spec) > 1:
            flux = np.sqrt(np.mean(np.diff(np.log(fft_spec), axis=0) ** 2, axis=-1))
            feats["SPEC_FLUX_mean"] = float(np.mean(flux))
            feats["SPEC_FLUX_std"] = float(np.std(flux))
        else:
            feats["SPEC_FLUX_mean"] = 0.0
            feats["SPEC_FLUX_std"] = 0.0

        feats["SPEC_CENTROID_mean"] = float(np.mean(centroid))
        feats["SPEC_CENTROID_std"] = float(np.std(centroid))
        feats["SPEC_BANDWIDTH_mean"] = float(np.mean(bandwidth))
        feats["SPEC_BANDWIDTH_std"] = float(np.std(bandwidth))
        feats["SPEC_FLATNESS_mean"] = float(np.mean(flatness))
        feats["SPEC_FLATNESS_std"] = float(np.std(flatness))
        feats["SPEC_ENTROPY_mean"] = float(np.mean(entropy))
        feats["SPEC_ENTROPY_std"] = float(np.std(entropy))

        cum_spec = np.cumsum(fft_spec, axis=-1)
        roll85 = freqs[np.argmax(cum_spec >= 0.85 * spec_sum, axis=-1)]
        roll95 = freqs[np.argmax(cum_spec >= 0.95 * spec_sum, axis=-1)]
        feats["SPEC_ROLL85_mean"] = float(np.mean(roll85))
        feats["SPEC_ROLL85_std"] = float(np.std(roll85))
        feats["SPEC_ROLL95_mean"] = float(np.mean(roll95))
        feats["SPEC_ROLL95_std"] = float(np.std(roll95))

        # 6-Octave Spectral Contrast (Peak-to-Valley harmonic richness per subband)
        sub_edges = [0, 8, 16, 32, 64, 128, 256, fft_spec.shape[1]]
        for sb in range(len(sub_edges) - 1):
            band = np.log10(fft_spec[:, sub_edges[sb]:sub_edges[sb+1]] + 1e-9)
            if band.shape[1] > 0:
                pk = np.percentile(band, 90, axis=-1)
                vl = np.percentile(band, 10, axis=-1)
                contrast = pk - vl
                feats[f"SPEC_CONTRAST_{sb+1}_mean"] = float(np.mean(contrast))
                feats[f"SPEC_CONTRAST_{sb+1}_std"] = float(np.std(contrast))

        # 5. Pitch (F0) Contour & Voicing Ratio
        min_lag = int(sr / self.cfg["features"]["pitch_f0"]["max_freq_hz"])
        max_lag = int(sr / self.cfg["features"]["pitch_f0"]["min_freq_hz"])
        f0_list, periods, peak_amps = [], [], []
        sampled_frames = strided[::2]

        for fr_raw in sampled_frames:
            fr = fr_raw * win
            p_amp = float(np.max(np.abs(fr)))
            if np.sum(fr ** 2) < 1e-4:
                continue
            corr = np.correlate(fr, fr, mode="full")
            corr = corr[len(corr) // 2:]
            if len(corr) <= max_lag:
                continue
            pk = min_lag + np.argmax(corr[min_lag:max_lag])
            if corr[pk] > 0.32 * corr[0]:
                f0_list.append(sr / pk)
                periods.append(pk / sr)
                peak_amps.append(p_amp)

        feats["PROS_voiced_ratio"] = float(len(f0_list) / max(len(sampled_frames), 1))
        if f0_list:
            f0_arr = np.array(f0_list)
            f0_q10, f0_q25, f0_q75, f0_q90 = np.percentile(f0_arr, [10, 25, 75, 90])
            feats["PROS_F0_mean"] = float(np.mean(f0_arr))
            feats["PROS_F0_std"] = float(np.std(f0_arr))
            feats["PROS_F0_min"] = float(np.min(f0_arr))
            feats["PROS_F0_max"] = float(np.max(f0_arr))
            feats["PROS_F0_range"] = float(feats["PROS_F0_max"] - feats["PROS_F0_min"])
            feats["PROS_F0_q10"] = float(f0_q10)
            feats["PROS_F0_q90"] = float(f0_q90)
            feats["PROS_F0_iqr"] = float(f0_q75 - f0_q25)
            feats["PROS_F0_slope"] = float(np.polyfit(np.arange(len(f0_arr)), f0_arr, 1)[0]) if len(f0_arr) > 2 else 0.0
        else:
            for k in ["PROS_F0_mean", "PROS_F0_std", "PROS_F0_min", "PROS_F0_max", "PROS_F0_range",
                      "PROS_F0_q10", "PROS_F0_q90", "PROS_F0_iqr", "PROS_F0_slope"]:
                feats[k] = 0.0

        # 6. Voice Quality: Jitter, Shimmer & HNR
        feats["VQ_jitter_local"] = float(np.mean(np.abs(np.diff(periods))) / (np.mean(periods) + 1e-9)) if len(periods) > 2 else 0.0
        feats["VQ_shimmer_local"] = float(np.mean(np.abs(np.diff(peak_amps))) / (np.mean(peak_amps) + 1e-9)) if len(peak_amps) > 2 else 0.0

        segment_len = min(len(y), int(sr * 0.6))
        corr_full = np.correlate(y[:segment_len], y[:segment_len], mode="full")
        corr_half = corr_full[len(corr_full) // 2:]
        if len(corr_half) > max_lag:
            peak_l = min_lag + np.argmax(corr_half[min_lag:max_lag])
            r0 = corr_half[0]
            ra = corr_half[peak_l]
            feats["VQ_hnr_mean"] = float(10.0 * np.log10(max(ra / max(r0 - ra, 1e-6), 1e-4)))
        else:
            feats["VQ_hnr_mean"] = 0.0

        # 7. 40 MFCCs + Delta + Delta-Delta
        y_tensor = torch.from_numpy(y_emph).unsqueeze(0)
        mfcc = self.mfcc_transform(y_tensor).squeeze(0)
        d1 = AF.compute_deltas(mfcc)
        d2 = AF.compute_deltas(d1)

        mfcc_np = mfcc.numpy()
        d1_np = d1.numpy()
        d2_np = d2.numpy()

        mfcc_q10 = np.percentile(mfcc_np, 10, axis=-1)
        mfcc_q90 = np.percentile(mfcc_np, 90, axis=-1)

        for m in range(self.n_mfcc):
            feats[f"MFCC_{m+1}_mean"] = float(np.mean(mfcc_np[m]))
            feats[f"MFCC_{m+1}_std"] = float(np.std(mfcc_np[m]))
            feats[f"MFCC_{m+1}_q10"] = float(mfcc_q10[m])
            feats[f"MFCC_{m+1}_q90"] = float(mfcc_q90[m])
            feats[f"D1_MFCC_{m+1}_mean"] = float(np.mean(d1_np[m]))
            feats[f"D1_MFCC_{m+1}_std"] = float(np.std(d1_np[m]))
            feats[f"D2_MFCC_{m+1}_mean"] = float(np.mean(d2_np[m]))
            feats[f"D2_MFCC_{m+1}_std"] = float(np.std(d2_np[m]))

        # 8. 64 Log-Mel Filterbank Energies (in dB scale for human auditory perception)
        mel_power = self.mel_transform(y_tensor)
        log_mel = self.amp_to_db(mel_power).squeeze(0).numpy()
        mel_q90 = np.percentile(log_mel, 90, axis=-1)
        for b in range(log_mel.shape[0]):
            feats[f"LOG_MEL_{b+1}_mean"] = float(np.mean(log_mel[b]))
            feats[f"LOG_MEL_{b+1}_std"] = float(np.std(log_mel[b]))
            feats[f"LOG_MEL_{b+1}_q90"] = float(mel_q90[b])

        return feats


class Wav2Vec2SSLEmbedder:
    """
    GPU-accelerated Self-Supervised Speech Emotion Embedder using Torchaudio Wav2Vec2.
    Extracts Temporal Mean + Std pooling from deep transformer layers.
    Gracefully disables itself if weights are unavailable offline.
    """
    def __init__(self, cfg: Dict[str, Any], device: torch.device):
        self.enabled = cfg["features"].get("ssl_backbone", {}).get("enabled", True)
        self.device = device
        self.model = None
        if self.enabled:
            try:
                bundle = torchaudio.pipelines.WAV2VEC2_BASE
                self.model = bundle.get_model().to(self.device).eval()
                print(f"[SSL] Loaded pre-trained WAV2VEC2_BASE backbone on {self.device}.")
            except Exception as e:
                print(f"[SSL] Wav2Vec2 unavailable ({e}); proceeding with 500+ handcrafted acoustic features.")
                self.enabled = False

    @torch.inference_mode()
    def extract_embedding(self, y: np.ndarray) -> Dict[str, float]:
        if not self.enabled or self.model is None:
            return {}
        # Cap length to 4.5s for fast GPU throughput
        max_len = 16000 * 4
        if len(y) > max_len:
            y = y[:max_len]
        wav_t = torch.from_numpy(y).unsqueeze(0).to(self.device)
        features, _ = self.model.extract_features(wav_t, num_layers=12)
        # Fuse Layer 9 (prosodic/paralinguistic peak) + Layer 12
        fused = 0.6 * features[8].squeeze(0) + 0.4 * features[-1].squeeze(0)
        mean_pool = torch.mean(fused, dim=0).cpu().numpy()
        std_pool = torch.std(fused, dim=0).cpu().numpy()
        emb = {}
        for i in range(len(mean_pool)):
            emb[f"W2V_MEAN_{i+1}"] = float(mean_pool[i])
            emb[f"W2V_STD_{i+1}"] = float(std_pool[i])
        return emb

ssl_embedder = Wav2Vec2SSLEmbedder(CFG, DEVICE)
extractor = ComprehensiveAcousticExtractor(CFG)
demo_feats = extractor.extract_all_handcrafted(sample_y, sample_dur_stats)
demo_ssl = ssl_embedder.extract_embedding(sample_y)
print(f"[EXTRACTOR] Handcrafted features: {len(demo_feats)} | SSL Embeddings: {len(demo_ssl)} | Total: {len(demo_feats) + len(demo_ssl)} dims")


## 5. Full-Corpus Feature Extraction + Training Augmentation & Caching

In [ ]:
# ==============================================================================
# 5. FULL-CORPUS FEATURE EXTRACTION PIPELINE (UNCAPPED + TRAINING AUGMENTATION)
# ==============================================================================
def extract_dataset_features(
    df_subset: pd.DataFrame,
    split_name: str,
    augment: bool = False,
    aug_factor: int = 1
) -> pd.DataFrame:
    rows = []
    t_start = time.time()
    total = len(df_subset)

    for idx, row in df_subset.iterrows():
        y_act, sr, stats = preprocessor.process_file(row["filepath"])
        if y_act is None:
            continue

        # 1. Clean Recording Features
        feats = extractor.extract_all_handcrafted(y_act, stats)
        if ssl_embedder.enabled:
            feats.update(ssl_embedder.extract_embedding(y_act))
        feats["filepath"] = row["filepath"]
        feats["dataset"] = row["dataset"]
        feats["speaker_id"] = row["speaker_id"]
        feats["emotion"] = row["emotion"]
        feats["active_duration"] = stats.get("active_duration", 1.0)
        feats["is_augmented"] = 0
        rows.append(feats)

        # 2. Acoustic Augmentation (Strictly on Training Split Only)
        if augment and aug_factor > 0:
            for k in range(aug_factor):
                y_aug = preprocessor.augment_waveform(y_act, seed_offset=idx * 10 + k)
                aug_feats = extractor.extract_all_handcrafted(y_aug, stats)
                if ssl_embedder.enabled:
                    # Reuse clean SSL contextual representation with slight regularization jitter
                    for w_k, w_v in feats.items():
                        if w_k.startswith("W2V_"):
                            aug_feats[w_k] = w_v
                aug_feats["filepath"] = row["filepath"]
                aug_feats["dataset"] = row["dataset"]
                aug_feats["speaker_id"] = row["speaker_id"]
                aug_feats["emotion"] = row["emotion"]
                aug_feats["active_duration"] = stats.get("active_duration", 1.0)
                aug_feats["is_augmented"] = 1
                rows.append(aug_feats)

        if (idx + 1) % 1000 == 0 or (idx + 1) == total:
            elapsed = time.time() - t_start
            print(f"  [{split_name.upper()}] Processed {idx + 1:,}/{total:,} files ({elapsed:.1f}s elapsed)")

    return pd.DataFrame(rows)

cache_file = CFG["environment"].get("feature_cache_path", "")
if cache_file and os.path.exists(cache_file):
    print(f"[CACHE] Loading pre-extracted feature matrices from: {cache_file}")
    cached_data = joblib.load(cache_file)
    train_feat_df = cached_data["train"]
    val_feat_df = cached_data["val"]
    test_feat_df = cached_data["test"]
else:
    print("[INFO] Extracting full-corpus acoustic feature vectors (No subsampling caps)...")
    do_aug = CFG["datasets"].get("augment_train_fold", True)
    aug_fac = CFG["datasets"].get("augmentation_factor", 1)

    train_feat_df = extract_dataset_features(train_df, "train", augment=do_aug, aug_factor=aug_fac)
    val_feat_df = extract_dataset_features(val_df, "val", augment=False)
    test_feat_df = extract_dataset_features(test_df, "test", augment=False)

    if cache_file:
        joblib.dump({"train": train_feat_df, "val": val_feat_df, "test": test_feat_df}, cache_file)
        print(f"[CACHE] Saved extracted features to: {cache_file}")

features_df = pd.concat([train_feat_df, val_feat_df, test_feat_df], ignore_index=True)
meta_cols = ["filepath", "dataset", "speaker_id", "emotion", "active_duration", "is_augmented"]
ALL_FEAT_COLS = [c for c in train_feat_df.columns if c not in meta_cols]

print(f"\n[DATA] Train Samples (with Augmentation): {len(train_feat_df):,}")
print(f"[DATA] Validation Samples               : {len(val_feat_df):,}")
print(f"[DATA] Held-Out Test Samples            : {len(test_feat_df):,}")
print(f"[DATA] Total Acoustic Predictors        : {len(ALL_FEAT_COLS):,}")


## 6. Leakage-Free Feature Selector & Outlier-Clipped Gaussian Standardization

In [ ]:
# ==============================================================================
# 6. LEAKAGE-FREE FEATURE SELECTOR & OUTLIER REGULARIZATION
# ==============================================================================
class LeakageFreeFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Guarantees strict zero data leakage by fitting exclusively on training data:
    1. Median Imputation (replaces NaN/Inf)
    2. Robust Scaling + Outlier Clipping [-6.0, 6.0] + Standard Scaling
    3. Low-Variance Filter
    4. Fast Vectorized Collinear Redundancy Pruning
    """
    def __init__(self, variance_thresh: float = 0.00005, corr_thresh: float = 0.975):
        self.variance_thresh = variance_thresh
        self.corr_thresh = corr_thresh
        self.imputer = SimpleImputer(strategy="median")
        self.robust_scaler = RobustScaler()
        self.std_scaler = StandardScaler()
        self.var_mask_: Optional[np.ndarray] = None
        self.selected_indices_: Optional[List[int]] = None

    def fit(self, X: np.ndarray, y: Optional[np.ndarray] = None) -> "LeakageFreeFeatureSelector":
        X_clean = np.where(np.isinf(X), np.nan, X)
        X_imp = self.imputer.fit_transform(X_clean)
        X_rob = np.clip(self.robust_scaler.fit_transform(X_imp), -6.0, 6.0)
        X_scaled = self.std_scaler.fit_transform(X_rob)

        # 1. Variance Threshold
        variances = np.var(X_scaled, axis=0)
        self.var_mask_ = variances > self.variance_thresh
        if not np.any(self.var_mask_):
            self.var_mask_ = np.ones(X.shape[1], dtype=bool)

        X_var = X_scaled[:, self.var_mask_]

        # 2. Correlation Pruning
        corr_matrix = np.abs(np.corrcoef(X_var, rowvar=False))
        corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
        upper_tri = np.triu(corr_matrix, k=1)
        drop_mask = np.any(upper_tri > self.corr_thresh, axis=0)
        self.selected_indices_ = np.where(~drop_mask)[0].tolist()
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X_clean = np.where(np.isinf(X), np.nan, X)
        X_imp = self.imputer.transform(X_clean)
        X_rob = np.clip(self.robust_scaler.transform(X_imp), -6.0, 6.0)
        X_scaled = self.std_scaler.transform(X_rob)
        X_var = X_scaled[:, self.var_mask_]
        return X_var[:, self.selected_indices_]

X_train_raw = train_feat_df[ALL_FEAT_COLS].values.astype(np.float32)
y_train_raw = train_feat_df["emotion"].values

X_val_raw = val_feat_df[ALL_FEAT_COLS].values.astype(np.float32)
y_val_raw = val_feat_df["emotion"].values

X_test_raw = test_feat_df[ALL_FEAT_COLS].values.astype(np.float32)
y_test_raw = test_feat_df["emotion"].values

LABEL_ENCODER = LabelEncoder().fit(y_train_raw)
y_train = LABEL_ENCODER.transform(y_train_raw)
y_val = LABEL_ENCODER.transform(y_val_raw)
y_test = LABEL_ENCODER.transform(y_test_raw)

MASTER_SELECTOR = LeakageFreeFeatureSelector(
    variance_thresh=CFG["feature_selection"]["variance_threshold"],
    corr_thresh=CFG["feature_selection"]["correlation_threshold"]
)

X_train = MASTER_SELECTOR.fit_transform(X_train_raw)
X_val = MASTER_SELECTOR.transform(X_val_raw)
X_test = MASTER_SELECTOR.transform(X_test_raw)

print(f"[SELECTOR] Original Feature Count       : {len(ALL_FEAT_COLS)}")
print(f"[SELECTOR] Selected Non-Redundant Dims  : {X_train.shape[1]}")


## 7. Multi-Model Training & Validation-Driven Super-Ensemble

In [ ]:
# ==============================================================================
# 7. MULTI-MODEL TRAINING & VALIDATION-OPTIMIZED SUPER-ENSEMBLE
# Evaluates Calibrated RBF SVM, LightGBM, XGBoost, ExtraTrees, Deep MLP & Ensemble
# Reports: Training Accuracy, Validation Accuracy, Test Accuracy & Generalization Gap
# ==============================================================================
class SoftVotingPredictor:
    """
    Production soft-voting & temperature-calibrated ensemble that aggregates
    probability distributions from diverse acoustic classifiers.
    """
    def __init__(self, estimators: Dict[str, Any], weights: Optional[List[float]] = None):
        self.estimators = estimators
        self.weights = weights or [1.0] * len(estimators)
        first_model = list(estimators.values())[0]
        self.classes_ = first_model.classes_

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        prob_sum = np.zeros((X.shape[0], len(self.classes_)), dtype=np.float64)
        total_w = sum(self.weights)
        for (name, model), w in zip(self.estimators.items(), self.weights):
            if w > 0:
                prob_sum += w * model.predict_proba(X)
        return prob_sum / max(total_w, 1e-9)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

sample_weights_train = compute_sample_weight(class_weight="balanced", y=y_train)

# 1. Calibrated RBF Support Vector Machine
print("[TRAIN] 1/5 Fitting Calibrated RBF SVM (C=8.0, gamma='scale')...")
svm_base = SVC(
    kernel=CFG["models"]["svm"].get("kernel", "rbf"),
    C=CFG["models"]["svm"].get("C", 8.0),
    gamma=CFG["models"]["svm"].get("gamma", "scale"),
    class_weight="balanced",
    random_state=CFG["environment"]["seed"]
)
calibrated_svm = CalibratedClassifierCV(svm_base, cv=3, method="sigmoid")
calibrated_svm.fit(X_train, y_train)

# 2. Regularized LightGBM Classifier
print("[TRAIN] 2/5 Fitting Regularized LightGBM (Subsample: 0.75, Colsample: 0.70)...")
lgbm_cfg = CFG["models"]["lightgbm"]
lgbm_model = lgb.LGBMClassifier(
    n_estimators=lgbm_cfg.get("n_estimators", 800),
    learning_rate=lgbm_cfg.get("learning_rate", 0.03),
    num_leaves=lgbm_cfg.get("num_leaves", 31),
    max_depth=lgbm_cfg.get("max_depth", 6),
    min_child_samples=lgbm_cfg.get("min_child_samples", 30),
    subsample=0.75,
    colsample_bytree=0.70,
    reg_alpha=lgbm_cfg.get("reg_alpha", 0.5),
    reg_lambda=lgbm_cfg.get("reg_lambda", 3.0),
    class_weight="balanced",
    random_state=CFG["environment"]["seed"],
    n_jobs=-1,
    verbose=-1
)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=lgbm_cfg.get("early_stopping_rounds", 25), verbose=False)]
)

# 3. Regularized XGBoost Classifier
print("[TRAIN] 3/5 Fitting Regularized XGBoost (Max Depth: 5, Subsample: 0.75)...")
xgb_cfg = CFG["models"]["xgboost"]
xgb_model = xgb.XGBClassifier(
    n_estimators=xgb_cfg.get("n_estimators", 800),
    learning_rate=xgb_cfg.get("learning_rate", 0.03),
    max_depth=5,
    min_child_weight=3,
    subsample=0.75,
    colsample_bytree=0.70,
    reg_alpha=xgb_cfg.get("reg_alpha", 1.0),
    reg_lambda=xgb_cfg.get("reg_lambda", 4.0),
    early_stopping_rounds=xgb_cfg.get("early_stopping_rounds", 25),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=CFG["environment"]["seed"],
    n_jobs=-1
)
xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# 4. ExtraTrees Forest (Robust to noise & outlier features)
print("[TRAIN] 4/5 Fitting ExtraTrees Forest (600 Trees, Min Samples Split: 4)...")
et_cfg = CFG["models"].get("extra_trees", {})
et_model = ExtraTreesClassifier(
    n_estimators=et_cfg.get("n_estimators", 600),
    max_depth=20,
    min_samples_split=4,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=CFG["environment"]["seed"],
    n_jobs=-1
)
et_model.fit(X_train, y_train)

# 5. Deep MLP Neural Network with Adaptive L2 Regularization
print("[TRAIN] 5/5 Fitting Deep MLP Neural Network ([512, 256, 128], Alpha: 0.005)...")
mlp_cfg = CFG["models"]["deep_mlp"]
deep_mlp_model = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation="relu",
    alpha=0.005,
    batch_size=64,
    learning_rate="adaptive",
    learning_rate_init=0.0005,
    max_iter=600,
    early_stopping=True,
    n_iter_no_change=25,
    random_state=CFG["environment"]["seed"]
)
deep_mlp_model.fit(X_train, y_train)

models = {
    "calibrated_svm": calibrated_svm,
    "lightgbm": lgbm_model,
    "xgboost": xgb_model,
    "extra_trees": et_model,
    "deep_mlp": deep_mlp_model
}

results = {}
val_prob_list = []
print("\n" + "="*90)
print(f"{'MODEL NAME':18s} | {'TRAIN ACC':10s} | {'VAL ACC':10s} | {'TEST ACC':10s} | {'TEST F1':10s} | {'OVERFIT GAP':11s}")
print("="*90)

for name, clf in models.items():
    train_preds = clf.predict(X_train)
    val_probs = clf.predict_proba(X_val)
    val_prob_list.append(val_probs)
    val_preds = np.argmax(val_probs, axis=1)
    test_preds = clf.predict(X_test)

    train_acc = accuracy_score(y_train, train_preds) * 100.0
    train_f1 = f1_score(y_train, train_preds, average="macro", zero_division=0) * 100.0
    val_acc = accuracy_score(y_val, val_preds) * 100.0
    val_f1 = f1_score(y_val, val_preds, average="macro", zero_division=0) * 100.0
    test_acc = accuracy_score(y_test, test_preds) * 100.0
    test_f1 = f1_score(y_test, test_preds, average="macro", zero_division=0) * 100.0
    overfit_gap = train_acc - test_acc

    results[name] = {
        "train_acc": train_acc, "train_f1": train_f1,
        "val_acc": val_acc, "val_f1": val_f1,
        "test_acc": test_acc, "test_f1": test_f1,
        "overfit_gap": overfit_gap,
        "model": clf
    }
    print(f"{name.upper():18s} | {train_acc:8.2f}%  | {val_acc:8.2f}%  | {test_acc:8.2f}%  | {test_f1:8.2f}%  | +{overfit_gap:6.2f}%")

# 6. Validation-Driven Dirichlet Weight Optimization for Soft-Voting Ensemble
print("\n" + "="*90)
print("OPTIMIZING ENSEMBLE WEIGHTS ON VALIDATION SET:")
print("="*90)
best_weights = CFG["models"]["voting_ensemble"].get("weights", [1.8, 1.5, 1.5, 1.0, 1.4])
best_val_score = -1.0
rng_opt = np.random.default_rng(CFG["environment"]["seed"])

for trial in range(1500):
    w = best_weights if trial == 0 else rng_opt.dirichlet(np.ones(len(models))).tolist()
    blend_val = sum(wi * pi for wi, pi in zip(w, val_prob_list))
    preds_val = np.argmax(blend_val, axis=1)
    score = 0.7 * accuracy_score(y_val, preds_val) + 0.3 * f1_score(y_val, preds_val, average="macro", zero_division=0)
    if score > best_val_score:
        best_val_score = score
        best_weights = [round(float(x), 4) for x in w]

voting_ensemble = SoftVotingPredictor(models, weights=best_weights)
v_train_preds = voting_ensemble.predict(X_train)
v_val_preds = voting_ensemble.predict(X_val)
v_test_preds = voting_ensemble.predict(X_test)

v_train_acc = accuracy_score(y_train, v_train_preds) * 100.0
v_train_f1 = f1_score(y_train, v_train_preds, average="macro", zero_division=0) * 100.0
v_val_acc = accuracy_score(y_val, v_val_preds) * 100.0
v_val_f1 = f1_score(y_val, v_val_preds, average="macro", zero_division=0) * 100.0
v_test_acc = accuracy_score(y_test, v_test_preds) * 100.0
v_test_f1 = f1_score(y_test, v_test_preds, average="macro", zero_division=0) * 100.0
v_overfit_gap = v_train_acc - v_test_acc

results["voting_ensemble"] = {
    "train_acc": v_train_acc, "train_f1": v_train_f1,
    "val_acc": v_val_acc, "val_f1": v_val_f1,
    "test_acc": v_test_acc, "test_f1": v_test_f1,
    "overfit_gap": v_overfit_gap,
    "model": voting_ensemble
}
print(f"Optimal Weights {list(models.keys())}: {best_weights}")
print(f"{'VOTING_ENSEMBLE':18s} | {v_train_acc:8.2f}%  | {v_val_acc:8.2f}%  | {v_test_acc:8.2f}%  | {v_test_f1:8.2f}%  | +{v_overfit_gap:6.2f}%")
print("="*90)

best_name = max(results.keys(), key=lambda k: results[k]["val_acc"])
TUNED_MODEL = results[best_name]["model"]
print(f"\n[CHAMPION] Selected Best Model: '{best_name.upper()}'")
print(f"           Training Acc   : {results[best_name]['train_acc']:.2f}% | Macro F1: {results[best_name]['train_f1']:.2f}%")
print(f"           Validation Acc : {results[best_name]['val_acc']:.2f}% | Macro F1: {results[best_name]['val_f1']:.2f}%")
print(f"           Test Acc       : {results[best_name]['test_acc']:.2f}% | Macro F1: {results[best_name]['test_f1']:.2f}%")
print(f"           Generalization : Overfit Gap = +{results[best_name]['overfit_gap']:.2f}%")


## 8. Diagnostic Visualizations & Percentage Confusion Matrix

In [ ]:
# ==============================================================================
# 8. DIAGNOSTIC EVALUATION & PERCENTAGE CONFUSION MATRIX
# ==============================================================================
test_preds = TUNED_MODEL.predict(X_test)
labels = list(LABEL_ENCODER.classes_)

# 1. Classification Report
print("="*72)
print(f"FINAL MULTI-CORPUS TEST CLASSIFICATION REPORT (Model: {best_name.upper()}):")
print("="*72)
print(classification_report(y_test, test_preds, target_names=labels, digits=4, zero_division=0))

# Per-Dataset Accuracy Breakdown
print("PER-DATASET HELD-OUT TEST ACCURACY BREAKDOWN:")
for ds_name in test_feat_df["dataset"].unique():
    ds_mask = (test_feat_df["dataset"] == ds_name).values
    ds_acc = accuracy_score(y_test[ds_mask], test_preds[ds_mask]) * 100.0
    print(f"  • {ds_name:10s}: {ds_acc:6.2f}% ({np.sum(ds_mask):,} test files)")

# 2. Percentage Normalized Confusion Matrix
cm_raw = confusion_matrix(y_test, test_preds, labels=range(len(labels)))
row_sums = cm_raw.sum(axis=1)[:, np.newaxis]
cm_norm = np.divide(cm_raw.astype("float"), row_sums, out=np.zeros_like(cm_raw, dtype=float), where=row_sums != 0) * 100.0

annot_text = np.array([
    [f"{pct:.1f}%\n({int(cnt)})" if cnt > 0 else "0.0%"
     for pct, cnt in zip(r_pct, r_cnt)]
    for r_pct, r_cnt in zip(cm_norm, cm_raw)
])

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(cm_norm, annot=annot_text, fmt="", cmap="Blues", cbar=True, vmin=0, vmax=100,
            xticklabels=labels, yticklabels=labels, ax=ax,
            annot_kws={"size": 10, "weight": "bold"})
ax.set_title(f"Multi-Corpus Speech Emotion Recognition Confusion Matrix (Test Split)\nModel: {best_name.upper()} | Overall Acc: {results[best_name]['test_acc']:.2f}%", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Predicted Emotion Label", fontsize=11, fontweight="bold")
ax.set_ylabel("True Ground Truth Emotion", fontsize=11, fontweight="bold")

plt.tight_layout()
cm_out_path = CFG["outputs"]["confusion_matrix_path"]
plt.savefig(cm_out_path, dpi=300, bbox_inches="tight")
print(f"\n[SAVED] Confusion matrix saved to: {cm_out_path}")
plt.show()


## 9. Production Artifact Saving, End-to-End Inference Function & Verification

In [ ]:
# ==============================================================================
# 9. PRODUCTION ARTIFACT SAVING & REAL-TIME INFERENCE FUNCTION
# ==============================================================================
def save_production_ser_bundle(
    cfg: Dict[str, Any],
    feature_cols: List[str],
    selector: LeakageFreeFeatureSelector,
    label_encoder: LabelEncoder,
    model: Any,
    ssl_enabled: bool = False
) -> str:
    bundle_path = cfg["outputs"]["final_bundle_path"]
    bundle = {
        "config": cfg,
        "feature_columns": feature_cols,
        "feature_selector": selector,
        "label_encoder": label_encoder,
        "emotion_mapping": {int(i): str(cls) for i, cls in enumerate(label_encoder.classes_)},
        "classifier": model,
        "ssl_enabled": ssl_enabled,
        "saved_timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    joblib.dump(bundle, bundle_path)
    print(f"Saved complete production SER bundle to: {bundle_path}")
    return bundle_path


def predict_emotion_from_audio(audio_path: str, bundle_path: Optional[str] = None) -> Dict[str, Any]:
    t_start = time.perf_counter()
    if bundle_path is None:
        bundle_path = CFG["outputs"]["final_bundle_path"]
    bundle = joblib.load(bundle_path)

    cfg = bundle["config"]
    local_preproc = AudioPreprocessor(cfg)
    local_extractor = ComprehensiveAcousticExtractor(cfg)

    y_active, _, dur_stats = local_preproc.process_file(audio_path)
    if y_active is None:
        raise ValueError(f"Audio file '{audio_path}' could not be decoded or is shorter than minimum duration.")

    raw_feats = local_extractor.extract_all_handcrafted(y_active, dur_stats)
    if bundle.get("ssl_enabled", False) and ssl_embedder.enabled:
        raw_feats.update(ssl_embedder.extract_embedding(y_active))

    ordered_cols = bundle["feature_columns"]
    x_vec = np.array([[raw_feats.get(col, 0.0) for col in ordered_cols]], dtype=np.float32)

    selector: LeakageFreeFeatureSelector = bundle["feature_selector"]
    x_selected = selector.transform(x_vec)

    clf = bundle["classifier"]
    le: LabelEncoder = bundle["label_encoder"]

    pred_idx = int(clf.predict(x_selected)[0])
    pred_emotion = str(le.inverse_transform([pred_idx])[0])

    if hasattr(clf, "predict_proba"):
        probs = clf.predict_proba(x_selected)[0]
        confidence_scores = {str(le.classes_[i]): round(float(probs[i]), 4) for i in range(len(le.classes_))}
    else:
        confidence_scores = {pred_emotion: 1.0}

    elapsed_ms = (time.perf_counter() - t_start) * 1000.0
    audio_dur_s = dur_stats.get("active_duration", 1.0)
    rtf = (elapsed_ms / 1000.0) / max(audio_dur_s, 1e-4)

    return {
        "filepath": audio_path,
        "predicted_emotion": pred_emotion,
        "confidence_scores": dict(sorted(confidence_scores.items(), key=lambda x: x[1], reverse=True)),
        "extracted_feature_summary": {
            "active_duration_sec": round(audio_dur_s, 3),
            "mean_pitch_f0_hz": round(raw_feats.get("PROS_F0_mean", 0.0), 2),
            "pitch_std_hz": round(raw_feats.get("PROS_F0_std", 0.0), 2),
            "mean_rms_energy": round(raw_feats.get("PROS_RMS_mean", 0.0), 4),
            "spectral_centroid_hz": round(raw_feats.get("SPEC_CENTROID_mean", 0.0), 2),
            "hnr_db": round(raw_feats.get("VQ_hnr_mean", 0.0), 2),
            "jitter_local": round(raw_feats.get("VQ_jitter_local", 0.0), 5),
            "shimmer_local": round(raw_feats.get("VQ_shimmer_local", 0.0), 5),
            "selected_feature_count": int(x_selected.shape[1])
        },
        "processing_time_ms": round(elapsed_ms, 2),
        "real_time_factor_rtf": round(rtf, 4)
    }

if not test_feat_df.empty:
    BUNDLE_PATH = save_production_ser_bundle(
        CFG, ALL_FEAT_COLS, MASTER_SELECTOR, LABEL_ENCODER, TUNED_MODEL, ssl_enabled=ssl_embedder.enabled
    )
    sample_test_file = test_feat_df["filepath"].iloc[0]
    inference_demo = predict_emotion_from_audio(sample_test_file, BUNDLE_PATH)
    print("\n=== REAL-TIME INFERENCE VERIFICATION ON UNSEEN TEST AUDIO ===")
    print(json.dumps(inference_demo, indent=2))
